<a href="https://colab.research.google.com/github/yussef862/blahblahblah/blob/main/Meat_Pipeline_V4_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report
print(f' TF: {tf.__version__}')

✅ TF: 2.20.0


In [ ]:
data_dir      = '/content/drive/MyDrive/combined_data'
meat_only_dir = '/content/meat_only'
meat_classes  = ['Beef', 'Buffalo meat', 'Camel meat', 'Goat meat', 'Lamb (sheep)']

os.makedirs(meat_only_dir, exist_ok=True)

for cls in meat_classes:
    src = os.path.join(data_dir, cls)
    dst = os.path.join(meat_only_dir, cls)
    if not os.path.exists(dst):
        shutil.copytree(src, dst)

print(' Classes:')
total = 0
for cls in sorted(os.listdir(meat_only_dir)):
    count = len(os.listdir(os.path.join(meat_only_dir, cls)))
    total += count
    print(f'  {cls}: {count} images')
print(f'Total: {total}')

📂 Classes:
  Beef: 297 images
  Buffalo meat: 300 images
  Camel meat: 299 images
  Goat meat: 300 images
  Lamb (sheep): 300 images
Total: 1496


In [ ]:
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

IMG_SIZE = (224, 224)
BATCH    = 16

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.6, 1.4],
    validation_split=0.2
)

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_data = train_gen.flow_from_directory(
    meat_only_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode='categorical',
    subset='training',
    seed=42
)

val_data = val_gen.flow_from_directory(
    meat_only_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

print('Classes:', train_data.class_indices)
print(f'Train: {train_data.samples} | Val: {val_data.samples}')

Found 1198 images belonging to 5 classes.
Found 298 images belonging to 5 classes.
Classes: {'Beef': 0, 'Buffalo meat': 1, 'Camel meat': 2, 'Goat meat': 3, 'Lamb (sheep)': 4}
Train: 1198 | Val: 298


In [ ]:
base = EfficientNetV2B0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base.trainable = False

x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
output = layers.Dense(5, activation='softmax')(x)

model = Model(inputs=base.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(' EfficientNetV2B0 ready')
print(f'Layers: {len(model.layers)}')

✅ EfficientNetV2B0 ready
Layers: 277


In [ ]:
callbacks_p1 = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        '/content/drive/MyDrive/ML Project /meat_model_v2b0.keras',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
]

print(' Phase 1: Training head...')
history_p1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks_p1
)

🚀 Phase 1: Training head...
Epoch 1/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.2488 - loss: 1.8808
Epoch 1: val_loss improved from None to 1.18385, saving model to /content/drive/MyDrive/ML Project /meat_model_v2b0.keras

Epoch 1: finished saving model to /content/drive/MyDrive/ML Project /meat_model_v2b0.keras
75/75 ━━━━━━━━━━━━━━━━━━━━ 198s 2s/step - accuracy: 0.3664 - loss: 1.5765 - val_accuracy: 0.7752 - val_loss: 1.1839 - learning_rate: 1.0000e-04
Epoch 2/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6000 - loss: 1.0200
Epoch 2: val_loss improved from 1.18385 to 0.84477, saving model to /content/drive/MyDrive/ML Project /meat_model_v2b0.keras

Epoch 2: finished saving model to /content/drive/MyDrive/ML Project /meat_model_v2b0.keras
75/75 ━━━━━━━━━━━━━━━━━━━━ 142s 2s/step - accuracy: 0.6269 - loss: 0.9634 - val_accuracy: 0.8624 - val_loss: 0.8448 - learning_rate: 1.0000e-04
Epoch 3/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7131 - loss: 0.7706
Epoc

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_p2 = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        '/content/drive/MyDrive/ML Project /meat_model_v2b0.keras',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
]

print(' Phase 2: Fine-tuning...')
history_p2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=callbacks_p2
)

🔓 Phase 2: Fine-tuning...
Epoch 1/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8661 - loss: 0.3612
Epoch 1: val_loss improved from None to 0.24432, saving model to /content/drive/MyDrive/ML Project /meat_model_v2b0.keras

Epoch 1: finished saving model to /content/drive/MyDrive/ML Project /meat_model_v2b0.keras
75/75 ━━━━━━━━━━━━━━━━━━━━ 198s 2s/step - accuracy: 0.8497 - loss: 0.3833 - val_accuracy: 0.9228 - val_loss: 0.2443 - learning_rate: 1.0000e-05
Epoch 2/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8500 - loss: 0.4173
Epoch 2: val_loss did not improve from 0.24432
75/75 ━━━━━━━━━━━━━━━━━━━━ 154s 2s/step - accuracy: 0.8556 - loss: 0.3881 - val_accuracy: 0.9128 - val_loss: 0.2507 - learning_rate: 1.0000e-05
Epoch 3/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8601 - loss: 0.3485
Epoch 3: val_loss improved from 0.24432 to 0.23844, saving model to /content/drive/MyDrive/ML Project /meat_model_v2b0.keras

Epoch 3: finished saving model to /content/drive/M

In [ ]:
val_data.reset()
preds  = model.predict(val_data)
y_pred = np.argmax(preds, axis=1)
y_true = val_data.classes
labels = list(val_data.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=labels))

19/19 ━━━━━━━━━━━━━━━━━━━━ 38s 2s/step
              precision    recall  f1-score   support

        Beef       0.83      0.98      0.90        59
Buffalo meat       0.98      0.95      0.97        60
  Camel meat       0.96      0.80      0.87        59
   Goat meat       0.94      0.98      0.96        60
Lamb (sheep)       0.98      0.95      0.97        60

    accuracy                           0.93       298
   macro avg       0.94      0.93      0.93       298
weighted avg       0.94      0.93      0.93       298



In [ ]:
import gradio as gr
import numpy as np
import tensorflow as tf
from PIL import Image
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

CLASS_NAMES = list(val_data.class_indices.keys())
CONFIDENCE_THRESHOLD = 0.65

def predict(image):
    if image is None:
        return "Please upload an image"

    img = Image.fromarray(image).resize((224, 224))
    img_array = np.array(img).astype(np.float32)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array, verbose=0)[0]
    confidence = float(np.max(preds))
    predicted = int(np.argmax(preds))

    if confidence < CONFIDENCE_THRESHOLD:
        return "Unknown / Not a meat"

    return f"{CLASS_NAMES[predicted]} ({confidence*100:.1f}%)"

interface = gr.Interface(
    fn=predict,
    inputs=gr.Image(label="Upload meat image"),
    outputs=gr.Textbox(label="Result"),
    title="Meat Classification - V4 Test"
)

interface.queue()
interface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://21b7dd5aff9e4561ad.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://21b7dd5aff9e4561ad.gradio.live


In [ ]:
def predict_debug(image):
    if image is None:
        return "Please upload an image"

    img = Image.fromarray(image).resize((224, 224))
    img_array = np.array(img).astype(np.float32)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array, verbose=0)[0]

    result = ""
    for i, cls in enumerate(CLASS_NAMES):
        result += f"{cls}: {preds[i]*100:.1f}%\n"

    result += f"\nMax confidence: {float(np.max(preds))*100:.1f}%"
    result += f"\nPredicted: {CLASS_NAMES[int(np.argmax(preds))]}"

    return result

interface2 = gr.Interface(
    fn=predict_debug,
    inputs=gr.Image(label="Upload meat image"),
    outputs=gr.Textbox(label="Debug Result"),
    title="Debug Mode"
)

interface2.queue()
interface2.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://798b5a0d39c2884ae6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://798b5a0d39c2884ae6.gradio.live


In [ ]:
model.save('/content/drive/MyDrive/ML Project /meat_model_v2b0.keras')
print(' Model saved!')

✅ Model saved!


In [ ]:
!pip install transformers -q

import torch
from transformers import CLIPProcessor, CLIPModel
import tensorflow as tf
import numpy as np
from PIL import Image
import gradio as gr
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as preprocess_v4

print('Loading CLIP...')
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print('Loading V3...')
model_v3 = tf.keras.models.load_model('/content/drive/MyDrive/ML Project /best_model_v3.keras')

print('V4 already in memory ')

CLASS_NAMES = ['Beef', 'Buffalo meat', 'Camel meat', 'Goat meat', 'Lamb (sheep)']
CLIP_THRESHOLD = 0.6
CONFIDENCE_THRESHOLD = 0.55

def predict_ensemble(image):
    if image is None:
        return "Please upload an image"

    pil_img = Image.fromarray(image).convert('RGB')

    inputs = clip_processor(
        text=["a photo of raw meat", "a photo that is not meat"],
        images=pil_img,
        return_tensors="pt",
        padding=True
    )
    with torch.no_grad():
        outputs   = clip_model(**inputs)
        probs     = outputs.logits_per_image.softmax(dim=1)
        meat_prob = float(probs[0][0])

    if meat_prob < CLIP_THRESHOLD:
        return "Unknown / Not a meat"

    img_v3 = pil_img.resize((224, 224))
    arr_v3 = np.array(img_v3) / 255.0
    arr_v3 = np.expand_dims(arr_v3, axis=0)
    preds_v3 = model_v3.predict(arr_v3, verbose=0)[0]

    img_v4 = pil_img.resize((224, 224))
    arr_v4 = np.array(img_v4).astype(np.float32)
    arr_v4 = preprocess_v4(arr_v4)
    arr_v4 = np.expand_dims(arr_v4, axis=0)
    preds_v4 = model.predict(arr_v4, verbose=0)[0]

    max_v3 = float(np.max(preds_v3))
    max_v4 = float(np.max(preds_v4))

    if max_v4 >= 0.90:
        final = preds_v4
        source = "V4"
    elif max_v3 >= 0.90:
        final = preds_v3
        source = "V3"
    else:
        final = (0.4 * preds_v3) + (0.6 * preds_v4)
        source = "Ensemble"

    confidence = float(np.max(final))
    predicted  = int(np.argmax(final))

    if confidence < CONFIDENCE_THRESHOLD:
        return "Unknown / Not a meat"

    return f"{CLASS_NAMES[predicted]} ({confidence*100:.1f}%) — {source}"

interface = gr.Interface(
    fn=predict_ensemble,
    inputs=gr.Image(label="Upload meat image"),
    outputs=gr.Textbox(label="Result"),
    title="Meat Classification — CLIP + V3 + V4",
    description="Stage 1: CLIP filters non-meat | Stage 2: V3 + V4 Ensemble"
)

interface.queue()
interface.launch(share=True, debug=True)

Loading CLIP...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Loading V3...
V4 already in memory ✅
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1dee755d1fd9eeac53.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1dee755d1fd9eeac53.gradio.live
